# Module 4 — Memory and RAG

**Course:** Build Your First AI Agent from Scratch  
**Community:** [Autonomous MSP](https://skool.com/autonomous-msp-2162)

**What you build in this module:**
- ChromaDB — a local vector database that persists to disk
- `index_incident()` — store a past incident with semantic search capability
- `find_similar_incidents()` — search by meaning, not keywords
- `get_context_for_query()` — format memory into a prompt-ready block

**Why it matters:** Without memory, your agent starts from zero on every ticket. With memory, it says: "BGP flapping — similar to INC-2341 from 14 months ago. Root cause was route-map. Similarity: 0.87."

## Step 1 — Install ChromaDB

ChromaDB is the only dependency for this module. It runs locally, stores to disk, and needs no server or Docker container.

In [ ]:
!pip install chromadb -q

## Step 2 — Create the Database

`PersistentClient` writes everything to disk. The `agent_memory_db` folder is created automatically in your working directory. You can shut down Python and come back — the indexed incidents are still there.

We use **cosine similarity** as the distance metric. Cosine measures the angle between two vectors, not their raw distance. For text, this is almost always the right choice — it handles different lengths of text gracefully.

In [ ]:
import chromadb

# PersistentClient → data survives between Python sessions
# Change the path if you want to store the DB somewhere specific
chroma_client = chromadb.PersistentClient(path="./agent_memory_db")

# Three collections — one for each type of knowledge
# get_or_create_collection is safe to run multiple times: won't overwrite existing data
incidents_collection = chroma_client.get_or_create_collection(
    name="incidents",
    metadata={"hnsw:space": "cosine"}
)

protocols_collection = chroma_client.get_or_create_collection(
    name="protocols",
    metadata={"hnsw:space": "cosine"}
)

standards_collection = chroma_client.get_or_create_collection(
    name="standards",
    metadata={"hnsw:space": "cosine"}
)

print("Collections created:")
print(f"  incidents  — for past tickets, root causes, resolutions")
print(f"  protocols  — for your specific network configs and topology notes")
print(f"  standards  — for your internal SOPs and runbooks")

## Step 3 — The index_incident() Function

This stores one incident into ChromaDB. Two things are important to understand:

**What gets embedded (searchable):** The `description` + `root_cause`. This is the "symptom + explanation" pair. When a new incident comes in with similar symptoms, this is what ChromaDB compares against.

**What goes in metadata (retrieved but not embedded):** `resolution`, `device`, `protocol`. These come back with search results but don't affect similarity scores.

`upsert` = insert or update. Safe to run multiple times — won't create duplicates.

In [ ]:
def index_incident(
    collection,
    incident_id: str,
    description: str,
    root_cause: str,
    resolution: str,
    device: str,
    protocol: str
):
    """
    Store one incident in ChromaDB.
    
    The 'document' (description + root_cause) is what gets
    embedded and searched. Everything else is metadata — it
    comes back with results but doesn't affect search scores.
    """
    # This is the text ChromaDB embeds and compares against new queries
    document = f"{description}. Root cause: {root_cause}"

    collection.upsert(
        ids=[incident_id],
        documents=[document],
        metadatas=[{
            "resolution": resolution,
            "device":     device,
            "protocol":   protocol,
        }]
    )
    print(f"  Indexed: {incident_id} ({protocol.upper()})")


print("index_incident() defined.")

## Step 4 — Seed 4 Real MSP Incidents

These are based on real failure patterns. Run this once. The data persists on disk — you don't need to re-run it.

Notice the specificity: each description uses protocol-specific language and real symptom details. This specificity is what makes semantic search accurate.

In [ ]:
seed_incidents = [
    {
        "incident_id": "INC-2341",
        "description": "BGP neighbor down on CE router after scheduled maintenance window",
        "root_cause":  "Route-map CUSTOMER-OUT had a deny-all entry added during pre-change review that was never removed",
        "resolution":  "Removed deny statement from route-map CUSTOMER-OUT, soft-reset BGP session, prefixes propagated within 90 seconds",
        "device":      "Cisco ASR1001-X",
        "protocol":    "bgp",
    },
    {
        "incident_id": "INC-1955",
        "description": "OSPF adjacency not forming between core switches after VLAN reconfiguration",
        "root_cause":  "OSPF area mismatch — one interface was moved to area 1 during VLAN change, neighbor was still in area 0",
        "resolution":  "Corrected area assignment on affected interface, adjacency formed within 40 seconds",
        "device":      "Nexus 9300",
        "protocol":    "ospf",
    },
    {
        "incident_id": "INC-2107",
        "description": "IPSec tunnel between branch and HQ intermittently dropping large file transfers",
        "root_cause":  "MTU mismatch on ISP handoff — ISP reduced path MTU to 1476, tunnel was not adjusting MSS clamping",
        "resolution":  "Applied ip tcp adjust-mss 1436 on tunnel interface both sides, set ip mtu 1476 on outside interface",
        "device":      "Cisco ISR 4331",
        "protocol":    "ipsec",
    },
    {
        "incident_id": "INC-2288",
        "description": "OSPF neighbor stuck in EXSTART state after firmware upgrade",
        "root_cause":  "Duplex mismatch introduced after firmware upgrade reset interface defaults — one side auto-negotiated to half-duplex",
        "resolution":  "Hardcoded duplex full and speed 1000 on both sides, OSPF adjacency recovered immediately",
        "device":      "HP Aruba 2930F",
        "protocol":    "ospf",
    },
]

print("Indexing incidents...")
for inc in seed_incidents:
    index_incident(incidents_collection, **inc)

print(f"\nTotal documents in collection: {incidents_collection.count()}")

## Step 5 — find_similar_incidents()

This is where the magic happens. You pass in a query string — the new incident description — and ChromaDB finds the closest past incidents by **meaning**, not keywords.

**Similarity score:** ChromaDB returns cosine distances (0=identical, 2=opposite). We convert to a 0.0–1.0 similarity score using `1 - (distance / 2)`.

- 1.0 = perfect match
- 0.85+ = highly similar
- 0.5–0.7 = related
- below 0.5 = probably not useful

In [ ]:
def find_similar_incidents(collection, query: str, n_results: int = 3) -> list:
    """
    Search for incidents similar to 'query' by meaning.
    Returns a list of dicts sorted by similarity (highest first).
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    incidents = []
    for i in range(len(results["ids"][0])):
        distance   = results["distances"][0][i]
        # ChromaDB cosine distance: 0=identical, 2=opposite
        # Convert to 0.0–1.0 similarity
        similarity = 1 - (distance / 2)

        incidents.append({
            "id":         results["ids"][0][i],
            "document":   results["documents"][0][i],
            "metadata":   results["metadatas"][0][i],
            "similarity": round(similarity, 3),
        })

    return incidents


# ── Test with 3 different queries ──────────────────────────────
test_queries = [
    "BGP session dropping after change window, no prefixes being advertised",
    "OSPF not coming up between two routers after network change",
    "VPN tunnel drops when transferring large files",
]

for query in test_queries:
    print(f"\nQuery: {query}")
    print("-" * 60)
    results = find_similar_incidents(incidents_collection, query)
    for r in results:
        print(f"  {r['id']} | Similarity: {r['similarity']:.3f} | {r['metadata']['protocol'].upper()}")
        print(f"  Resolution: {r['metadata']['resolution'][:80]}...")

## Step 6 — The 0.5 Threshold Filter

Not every query will have a good match. If the agent retrieves a match with similarity 0.3 and presents it as relevant history, that is **worse** than no memory — it is fabricated context with a database behind it.

Always filter results below your minimum threshold before injecting them into the prompt.

In [ ]:
# ChromaDB's default embedding model (all-MiniLM) gives ~0.55 baseline
# similarity even for unrelated content. Start at 0.65 — raise to 0.75
# as your knowledge base grows and incidents become more specific.
SIMILARITY_THRESHOLD = 0.65


def filter_relevant_incidents(incidents: list) -> list:
    """Keep only incidents above the similarity threshold."""
    return [inc for inc in incidents if inc["similarity"] >= SIMILARITY_THRESHOLD]


# Test it with a query that has no good match
poor_query = "coffee machine broken in the office kitchen"
raw = find_similar_incidents(incidents_collection, poor_query)
filtered = filter_relevant_incidents(raw)

print(f"Query: '{poor_query}'")
print(f"Raw results: {len(raw)}  |  After filtering: {len(filtered)}")
for r in raw:
    status = "KEPT" if r["similarity"] >= SIMILARITY_THRESHOLD else "FILTERED OUT"
    print(f"  {r['id']} | Similarity: {r['similarity']:.3f} | {status}")

## Step 7 — get_context_for_query()

This is the bridge between your memory layer and your agent's prompt.

It takes a query, retrieves + filters relevant incidents, and formats them into a text block the LLM can read. This block gets prepended to the system prompt or appended to the user message. That is the complete RAG pipeline: retrieve → filter → format → inject.

In [ ]:
def get_context_for_query(collection, query: str) -> str:
    """
    Full RAG pipeline:
    1. Search ChromaDB for similar incidents
    2. Filter by similarity threshold
    3. Format as a text block for the LLM prompt
    """
    raw_results = find_similar_incidents(collection, query, n_results=3)
    relevant    = filter_relevant_incidents(raw_results)

    if not relevant:
        return "No similar incidents found in knowledge base."

    context_lines = ["RELEVANT PAST INCIDENTS:"]
    for inc in relevant:
        meta = inc["metadata"]
        context_lines.append(
            f"\n[{inc['id']}] Similarity: {inc['similarity']}\n"
            f"Incident: {inc['document']}\n"
            f"Resolution: {meta['resolution']}\n"
            f"Device: {meta['device']} | Protocol: {meta['protocol']}"
        )

    return "\n".join(context_lines)


def build_agent_prompt(query: str, context: str) -> str:
    """Wrap the RAG context into the full agent prompt."""
    return f"""You are a network troubleshooting assistant for an MSP.

{context}

Current incident: {query}

Based on past incidents above (if relevant), provide a diagnosis and recommended resolution steps."""


# ── Demo the full pipeline ─────────────────────────────────────
demo_query = "BGP neighbor dropped after maintenance, routes not showing in table"
context    = get_context_for_query(incidents_collection, demo_query)
prompt     = build_agent_prompt(demo_query, context)

print("=== PROMPT THAT WOULD BE SENT TO THE LLM ===")
print(prompt)

## Step 8 — What NOT to Index

More data is not always better. Run this cell to see the difference between useful and noisy indexing.

In [ ]:
# ── What belongs in your knowledge base ────────────────────────

GOOD_TO_INDEX = [
    "Your specific client incidents with exact symptoms and device details",
    "Firmware-specific bugs you have hit on real hardware",
    "Your IP addressing scheme and device naming conventions",
    "Your internal SOPs — the 6-step OSPF adjacency check your senior engineer wrote",
    "Vendor quirks specific to your client environments",
]

DO_NOT_INDEX = [
    "General networking theory — BGP path selection, OSPF LSA types, STP algorithm",
    "Generic vendor config guides from Cisco documentation",
    "Anything that applies equally to every network on earth",
    "Information the LLM already knows from training",
]

print("Index this (specific to YOUR environment):")
for item in GOOD_TO_INDEX:
    print(f"  + {item}")

print("\nDo NOT index (LLM already knows this):")
for item in DO_NOT_INDEX:
    print(f"  - {item}")

print("\nRule: if a junior engineer at a DIFFERENT MSP would benefit → LLM already knows it.")
print("      If it's specific to your clients and history → put it in ChromaDB.")

## What's Next

Your agent now has memory. In **Module 5** you replace the simple for-loop with LangGraph — a proper state machine that makes your agent's behavior explicit, traceable, and production-ready.

---

**Module Challenge:** Index 3 real incidents from your ticketing system. Then run a query with a slightly different description of the same problem (the way a different engineer might write it). Post your similarity score in **#agent-builds**.